# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PalSoham/flyrank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring (Lane 2)  
**Goal:** Build the full feature vector, audit every column for leakage, and confirm privacy safety before any modeling.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.preprocessing import LabelEncoder

for p in ['../../data/raw/content_refresh_anonymized.csv',
          'data/raw/content_refresh_anonymized.csv']:
    if os.path.exists(p):
        CSV_PATH = p; break

raw = pd.read_csv(CSV_PATH)
print(f'Loaded starter CSV: {raw.shape}')
print(f'Columns: {len(raw.columns)}')


Loaded starter CSV: (30000, 44)
Columns: 44


## 1. Build the feature vector

The feature vector for Lane 2 follows the starter pipeline's logic from `scripts/01_prepare_features.py`, extended with explicit missingness flags to handle patterned gaps by content type.

**Engineering decisions:**
- Log-transform heavy-tailed count columns (`impressions_90d`, `clicks_90d`, `sessions_90d`) to reduce the influence of outlier pages
- Replace `search_volume = NaN` with `0` AND add a `has_keyword_data` binary flag — this separates 'no keyword data' from 'zero search volume'
- Replace `avg_position = 0` with the column median (0 means 'no data', not rank zero) AND add a `has_position` flag
- Categorical columns are label-encoded (for random forest) — will need one-hot encoding for linear models
- All features are trailing-90d or static properties — none requires future information

In [2]:
# ── Prepare the feature-engineering pipeline ────────────────────────────────

df = raw.copy()

# 1. Filter: match starter pipeline criteria
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset='content_id').reset_index(drop=True)
print(f'After filter: {len(df):,} rows')

# 2. Label — NEVER a feature
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# 3. Log-transform heavy-tailed counts
for col in ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d']:
    df[f'log_{col}'] = np.log1p(df[col].fillna(0))

# 4. Missingness flags (avoids encoding content_type silently via fillna)
df['has_keyword_data'] = df['search_volume'].notna().astype(int)
df['has_position'] = (df['avg_position'] > 0).astype(int)
df['has_word_count'] = df['word_count'].notna().astype(int)

# 5. Impute: numeric fill with median (or 0 for counts)
med_position = df[df['avg_position'] > 0]['avg_position'].median()
df['avg_position_clean'] = df['avg_position'].replace(0, np.nan).fillna(med_position)
for col in ['search_volume', 'competition', 'cpc', 'word_count', 'char_count',
            'scroll_rate', 'engagement_rate', 'ctr']:
    df[col] = df[col].fillna(0)

# 6. Categorical encode
cat_cols = ['content_type', 'main_intent', 'age_tier', 'freshness_tier',
            'word_count_tier', 'impression_tier', 'position_tier']
for col in cat_cols:
    df[col] = df[col].fillna('unknown')
    df[f'{col}_enc'] = LabelEncoder().fit_transform(df[col])

# 7. Define final feature set
NUMERIC_FEATURES = [
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d',
    'days_with_impressions', 'days_with_sessions',
    'avg_position_clean', 'ctr',
    'content_age_days', 'days_since_last_update',
    'word_count', 'char_count',
    'engagement_rate', 'scroll_rate',
    'search_volume', 'competition', 'cpc',
    'has_keyword_data', 'has_position', 'has_word_count',
]
CAT_ENC_FEATURES = [f'{c}_enc' for c in cat_cols]
ALL_FEATURES = NUMERIC_FEATURES + CAT_ENC_FEATURES

print(f'Total features: {len(ALL_FEATURES)}')
print(f'Numeric: {len(NUMERIC_FEATURES)}, Categorical-encoded: {len(CAT_ENC_FEATURES)}')
print(f'Positive rate (label): {df["is_declining_label"].mean():.3f}')
print()
print('Feature list:')
for f in ALL_FEATURES:
    print(f'  {f}')


After filter: 30,000 rows
Total features: 26
Numeric: 19, Categorical-encoded: 7
Positive rate (label): 0.542

Feature list:
  log_impressions_90d
  log_clicks_90d
  log_sessions_90d
  days_with_impressions
  days_with_sessions
  avg_position_clean
  ctr
  content_age_days
  days_since_last_update
  word_count
  char_count
  engagement_rate
  scroll_rate
  search_volume
  competition
  cpc
  has_keyword_data
  has_position
  has_word_count
  content_type_enc
  main_intent_enc
  age_tier_enc
  freshness_tier_enc
  word_count_tier_enc
  impression_tier_enc
  position_tier_enc
  ctr


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `log_impressions_90d` | log1p of 90d GSC impressions — volume proxy | never missing (filter ensures >0) | before decision — trailing 90d aggregate |
| `log_clicks_90d` | log1p of 90d GSC clicks | 0-filled (no clicks is valid) | before decision — trailing 90d aggregate |
| `log_sessions_90d` | log1p of 90d GA4 sessions | 0-filled | before decision — trailing 90d aggregate |
| `days_with_impressions` | days in 90d window with >=1 impression (0-90) | never missing | before decision — trailing aggregate |
| `days_with_sessions` | days in 90d window with >=1 session (0-90) | 0-filled | before decision — trailing aggregate |
| `avg_position_clean` | mean GSC rank, 0 replaced with column median | median-imputed; `has_position` flag added | before decision — trailing aggregate |
| `ctr` | clicks/impressions x100 in 90d window | 0-filled | before decision — trailing aggregate |
| `content_age_days` | days since page creation | never missing in this slice (filter: >=90) | always available — static page property |
| `days_since_last_update` | days since last content edit | never missing | always available — static page property |
| `word_count` | article word count | 0-filled; `has_word_count` flag added | available before decision — static property |
| `char_count` | article character count | 0-filled alongside word_count | available before decision — static property |
| `engagement_rate` | engaged_sessions/sessions x100 | 0-filled when sessions=0 | before decision — trailing aggregate |
| `scroll_rate` | scroll_events/pageviews x100 (can exceed 100) | 0-filled when pageviews=0 | before decision — trailing aggregate |
| `search_volume` | keyword search volume estimate | 0-filled; `has_keyword_data` flag added | before decision — from keyword metadata |
| `competition`, `cpc` | keyword competitiveness signals | 0-filled with flag | before decision — from keyword metadata |
| `has_keyword_data` | 1 if search_volume was present, 0 if missing (avoids encoding content_type) | never missing | always available |
| `has_position`, `has_word_count` | availability flags for position and word count | never missing | always available |
| `content_type_enc`, `main_intent_enc` etc. | label-encoded tier/category columns | 'unknown' fill | always available — static metadata |

### Key gotchas
- `avg_position = 0` means **no data**, not rank zero (1,205 rows). Using it raw would make those pages look like they rank #0 — which doesn't exist.
- `ctr = 0.76` means **0.76%**, not 76%. All rate columns are ×100 percentages.
- `scroll_rate` and `ai_traffic_pct` can exceed 100 due to cross-system measurement.
- Missingness in keyword columns follows `content_type` — `feedly article` rows have 100% missing search_volume. Using `fillna(0)` without a flag silently encodes content_type.

In [3]:
# ── Verify feature completeness and missingness flags ───────────────────────
print('=== Missingness in final feature matrix ===')
X = df[ALL_FEATURES]
null_counts = X.isnull().sum()
if null_counts.sum() == 0:
    print('No missing values in feature matrix after engineering.')
else:
    print('Missing values found (should be 0 after engineering):')
    print(null_counts[null_counts > 0])
print()

# Verify the has_keyword_data flag aligns with content_type
print('=== has_keyword_data by content_type ===')
check = df.groupby('content_type')['has_keyword_data'].mean().round(3)
print(check.to_string())
print()
print('has_keyword_data=0.0 for feedly article confirms the flag captures the pattern.')
print('Models can now learn from both groups without content_type leaking in silently.')
print()

# Verify has_position flag
print('=== has_position by position_tier ===')
pos_check = df.groupby('position_tier')['has_position'].mean()
print(pos_check.to_string())


=== Missingness in final feature matrix ===
No missing values in feature matrix after engineering.

=== has_keyword_data by content_type ===
content_type
comparison article    1.000
feedly article        0.000
keyword article       1.000
Name: has_keyword_data, dtype: float64

has_keyword_data=0.0 for feedly article confirms the flag captures the pattern.
Models can now learn from both groups without content_type leaking in silently.

=== has_position by position_tier ===
position_tier
deep        1.0
no_data     0.0
page_1      1.0
page_3_5    1.0
striking    1.0
top_3       1.0
Name: has_position, dtype: float64


## 3. The leakage hunt

**Attack the feature set systematically.** For each leakage category, run a test.

### Leakage taxonomy for Lane 2

| Risk | Columns | Status |
|---|---|---|
| Label-derived features | `trend_direction`, `trend_pct`, `impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`, `sessions_last_30d`, `clicks_prev_30d`, `sessions_prev_30d` | **EXCLUDED** — not in feature list |
| Future-window overlap | any column computed from data after the decision point | **NONE** — all features are trailing 90d or static |
| Product decision flags | `health_score`, `priority_score`, `action_type`, `refresh_tier` | **NOT IN DATASET** — deliberately removed before release |
| Privacy / raw text | raw URL, domain, query, title, client name | **NOT IN DATASET** — scrambled before release |

### The deliberate leakage test

Add `trend_pct` to the features (it directly computes the label), train, measure the score jump, then remove it. The score jump IS the confession of leakage.

In [4]:
# ── Baseline: honest feature set ────────────────────────────────────────────
X_clean = df[ALL_FEATURES].values
y = df['is_declining_label'].values
groups = LabelEncoder().fit_transform(df['client_id'])

cv = GroupKFold(n_splits=5)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

scores_clean = cross_val_score(rf, X_clean, y, cv=cv, groups=groups,
                                scoring='average_precision')
print('=== HONEST model (no leaky features) ===')
print(f'Avg Precision: {scores_clean.mean():.3f} (+/- {scores_clean.std():.3f})')
print(f'Base rate: {y.mean():.3f}  (naive baseline = {y.mean():.3f})')
print()

# ── Deliberate leak 1: add trend_pct (direct label source) ──────────────────
df['trend_pct_leak'] = df['trend_pct'].fillna(0)
X_leak1 = np.hstack([X_clean, df[['trend_pct_leak']].values])
scores_leak1 = cross_val_score(rf, X_leak1, y, cv=cv, groups=groups,
                                scoring='average_precision')
print('=== LEAKY model 1: + trend_pct (direct label source) ===')
print(f'Avg Precision: {scores_leak1.mean():.3f} (+/- {scores_leak1.std():.3f})')
print(f'Jump vs honest: +{scores_leak1.mean()-scores_clean.mean():.3f}')
print()

# ── Deliberate leak 2: add impressions_last_30d (label source sibling) ──────
df['last30_leak'] = df['impressions_last_30d'].fillna(0)
X_leak2 = np.hstack([X_clean, df[['last30_leak']].values])
scores_leak2 = cross_val_score(rf, X_leak2, y, cv=cv, groups=groups,
                                scoring='average_precision')
print('=== LEAKY model 2: + impressions_last_30d (label source sibling) ===')
print(f'Avg Precision: {scores_leak2.mean():.3f} (+/- {scores_leak2.std():.3f})')
print(f'Jump vs honest: +{scores_leak2.mean()-scores_clean.mean():.3f}')
print()

print('=== REMOVING all leaky columns ===')
df.drop(columns=['trend_pct_leak', 'last30_leak'], errors='ignore', inplace=True)
print(f'Honest Avg Precision (KEEP THIS): {scores_clean.mean():.3f}')
print(f'Leak-1 Avg Precision (DISCARDED): {scores_leak1.mean():.3f}')
print(f'Leak-2 Avg Precision (DISCARDED): {scores_leak2.mean():.3f}')
print()
print('CONCLUSION: Both leaks produce near-perfect scores because trend_pct and')
print('impressions_last_30d encode the label directly. The honest model at')
print(f'{scores_clean.mean():.3f} Avg Precision is the starting point to beat.')


=== HONEST model (no leaky features) ===
Avg Precision: 0.558 (+/- 0.042)
Base rate: 0.542  (naive baseline = 0.542)

=== LEAKY model 1: + trend_pct (direct label source) ===
Avg Precision: 0.973 (+/- 0.009)
Jump vs honest: +0.415

=== LEAKY model 2: + impressions_last_30d (label source sibling) ===
Avg Precision: 0.918 (+/- 0.021)
Jump vs honest: +0.360

=== REMOVING all leaky columns ===
Honest Avg Precision (KEEP THIS): 0.558
Leak-1 Avg Precision (DISCARDED): 0.973
Leak-2 Avg Precision (DISCARDED): 0.918

CONCLUSION: Both leaks produce near-perfect scores because trend_pct and
impressions_last_30d encode the label directly. The honest model at
0.558 Avg Precision is the starting point to beat.


### Top feature importance sanity check

If leakage is present, one feature will tower over all others with near-100% importance. Run the honest model's feature importances to confirm no single feature dominates pathologically.

In [5]:
# ── Fit once on full data to inspect feature importance ─────────────────────
# (informational only — real evaluation uses CV)
rf_full = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_full.fit(X_clean, y)

importances = sorted(zip(ALL_FEATURES, rf_full.feature_importances_),
                     key=lambda x: x[1], reverse=True)

print('=== Top-10 feature importances (honest model) ===')
print(f'{"Feature":<28} {"Importance":>10}  Bar')
print('-' * 60)
for feat, imp in importances[:10]:
    bar = '#' * int(imp * 100)
    print(f'{feat:<28} {imp:>10.4f}  {bar}')
print()
top1_imp = importances[0][1]
print(f'Top feature importance: {top1_imp:.4f}')
if top1_imp > 0.5:
    print('WARNING: one feature dominates — investigate for leakage')
else:
    print('OK: no single feature dominates (top feature < 50% importance)')
    print('Importance is distributed across multiple signals — pattern is genuinely multi-signal.')


=== Top-10 feature importances (honest model) ===
Feature                      Importance  Bar
------------------------------------------------------------
days_with_impressions             0.1592  ###############
log_impressions_90d               0.1304  #############
avg_position_clean                0.1087  ##########
content_age_days                  0.0963  #########
log_clicks_90d                    0.0418  ####
char_count                        0.0401  ####
word_count                        0.0383  ###
ctr                               0.0341  ###
scroll_rate                       0.0318  ###
days_with_sessions                0.0275  ##

Top feature importance: 0.1592
OK: no single feature dominates (top feature < 50% importance)
Importance is distributed across multiple signals — pattern is genuinely multi-signal.


## 4. What I excluded and why

| Field | Why excluded |
|---|---|
| `trend_direction` | **Label source.** The proxy label is derived from this column. Using it as a feature means the model learns the label's own definition — circular. |
| `trend_pct` | **Label source.** Raw percentage that computes `trend_direction`. Including it causes Avg Precision to jump from 0.558 to 0.973 — a confirmed leakage confession. |
| `impressions_last_30d` | **Label source sibling.** Numerator of the trend calculation. Including it alone pushes Avg Precision from 0.558 to 0.918. |
| `impressions_prev_30d` | **Label source sibling.** Denominator of the trend calculation. |
| `clicks_last_30d`, `clicks_prev_30d` | **Same window as label.** Even though clicks don't directly compute the label, their 30d window overlaps — a future-window leakage risk. |
| `sessions_last_30d`, `sessions_prev_30d` | **Same window as label.** Same reasoning as clicks. |
| `is_declining_label` (if accidentally included) | **IS the label.** Would cause 100% leakage by definition. |
| `provider_used`, `model_used` | **Product metadata.** Which LLM generated the article — a production decision, not an observable content signal. Could cause data leakage if the LLM choice correlates with client-level strategies. |
| `ai_traffic_pct`, `ai_sessions_90d` | **Sparse signal** (~2% of rows have any AI sessions). Including as a main feature would introduce strong systematic missingness by content type and client. Excluded from the primary feature set; can be studied separately. |
| `content_id`, `client_id` | **Context identifiers.** Pseudonymous hash codes — useful for grouping, deduplication, and train/test splitting, but they encode no learnable signal about page quality. |
| Raw URL, domain, query, title, client name | **Not in dataset** (removed before release). Stated for completeness — these would be privacy violations if present. |

In [6]:
# ── Confirm excluded columns are not in ALL_FEATURES ───────────────────────
EXCLUDED = [
    'trend_direction', 'trend_pct',
    'impressions_last_30d', 'impressions_prev_30d',
    'clicks_last_30d', 'clicks_prev_30d',
    'sessions_last_30d', 'sessions_prev_30d',
    'is_declining_label',
    'provider_used', 'model_used',
    'content_id', 'client_id',
]

violations = [col for col in EXCLUDED if col in ALL_FEATURES]
print('=== Exclusion audit ===')
if violations:
    print(f'VIOLATION: these excluded columns appear in feature list: {violations}')
else:
    print('PASS: no excluded columns in ALL_FEATURES')

print()
print('Excluded columns confirmed absent from feature matrix:')
for col in EXCLUDED:
    in_feat = col in ALL_FEATURES
    status = 'IN FEATURES (BUG)' if in_feat else 'correctly excluded'
    print(f'  {col:<32}: {status}')


=== Exclusion audit ===
PASS: no excluded columns in ALL_FEATURES

Excluded columns confirmed absent from feature matrix:
  trend_direction                 : correctly excluded
  trend_pct                       : correctly excluded
  impressions_last_30d            : correctly excluded
  impressions_prev_30d            : correctly excluded
  clicks_last_30d                 : correctly excluded
  clicks_prev_30d                 : correctly excluded
  sessions_last_30d               : correctly excluded
  sessions_prev_30d               : correctly excluded
  is_declining_label              : correctly excluded
  provider_used                   : correctly excluded
  model_used                      : correctly excluded
  content_id                      : correctly excluded
  client_id                       : correctly excluded


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Feature vector built with 26 columns, all trailing-90d or static — no future information
- [x] has_keyword_data and has_position flags added to handle patterned missingness
- [x] Two deliberate leakage tests run (trend_pct, impressions_last_30d); both confessed with large score jumps; both removed
- [x] Top feature importance sanity check passed (no single feature > 50%)
- [x] Exclusion audit confirmed all excluded columns absent from feature matrix
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.